# 05 — Machine Learning, MLflow & Unity Catalog Model Registry: Real Estate (Property Listings, Market Valuation & AVM Pricing)

**ML Task:** Regression — Automated Valuation Model (AVM) Property Price Prediction

**Purpose:** Engineer features into `real_estate_prod.ml.real_estate_avm_features`, train and track the model with MLflow, register into Unity Catalog (`real_estate_prod.ml.property_avm_price_model`), assign the `@champion` alias, and run batch inference into `real_estate_prod.gold.gold_property_avm_predictions`.

In [ ]:
# Databricks Widget Parameters (overridden by Lakeflow Jobs / Databricks Asset Bundles)
dbutils.widgets.text("catalog_name", "real_estate_prod", "Unity Catalog Name")
CATALOG = dbutils.widgets.get("catalog_name")
VOLUME_PATH = f"/Volumes/{CATALOG}/bronze/raw_landing"
CHECKPOINT_PATH = f"/Volumes/{CATALOG}/bronze/checkpoints"
print(f"Active Catalog: {CATALOG} | Landing Volume: {VOLUME_PATH}")

## Step 1: Configure MLflow Experiment Path

In [ ]:
import mlflow
current_user = spark.sql("SELECT current_user()").first()[0]
exp_path = f"/Users/{current_user}/real_estate_experiments"
mlflow.set_experiment(exp_path)
print(f"MLflow Experiment set to: {exp_path}")

## Step 2: Feature Engineering, Training, Unity Catalog Registration & Batch Scoring

In [ ]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from pyspark.sql import functions as F

props = spark.table(f"{CATALOG}.silver.silver_properties_clean")
features_df = (
    props.select(
        "property_id", "sqft", "bedrooms", "bathrooms", "property_age_years", "school_rating", "days_on_market",
        F.when(F.col("city") == "Seattle", 1).otherwise(0).alias("city_seattle"),
        F.when(F.col("city") == "Miami", 1).otherwise(0).alias("city_miami"),
        F.when(F.col("city") == "Austin", 1).otherwise(0).alias("city_austin"),
        F.col("list_price_usd").alias("target_price_usd"),
    )
)
features_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.ml.real_estate_avm_features")

pdf = features_df.toPandas().dropna()
FEATURE_COLS = ["sqft", "bedrooms", "bathrooms", "property_age_years", "school_rating", "days_on_market", "city_seattle", "city_miami", "city_austin"]
X, y = pdf[FEATURE_COLS], pdf["target_price_usd"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

mlflow.set_registry_uri("databricks-uc")
model_name = f"{CATALOG}.ml.property_avm_price_model"

with mlflow.start_run(run_name="real_estate_rf_avm"):
    model = RandomForestRegressor(n_estimators=150, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    mlflow.log_params({"n_estimators": 150, "max_depth": 10, "domain": "real_estate"})
    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})

    signature = infer_signature(X_train, preds)
    mlflow.sklearn.log_model(model, artifact_path="model", signature=signature, input_example=X_train.head(3), registered_model_name=model_name)
    print(f"Registered {model_name} | RMSE=${rmse:,.0f}, MAE=${mae:,.0f}, R2={r2:.3f}")

client = mlflow.MlflowClient()
latest_ver = client.get_registered_model(model_name).latest_versions[-1].version
client.set_registered_model_alias(model_name, "champion", latest_ver)

loaded = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")
pdf["predicted_price_usd"] = loaded.predict(pdf[FEATURE_COLS]).round(0)
spark.createDataFrame(pdf[["property_id", "target_price_usd", "predicted_price_usd"]]).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.gold.gold_property_avm_predictions")
print("Batch predictions saved to gold.gold_property_avm_predictions")

## Step 3: Verify Batch Prediction Output in Gold Layer

In [ ]:
display(spark.table(f"{CATALOG}.gold.gold_property_avm_predictions").limit(15))